https://arxiv.org/abs/1608.06993

In [21]:
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary
import numpy as np
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torch.optim import Adam
from sklearn.metrics import accuracy_score
import math

In [22]:
data = pd.read_csv("../data/chest_xray/chest_xray_dataset.csv")

In [23]:
train_data = data[data['split'] == 'train']
train_data.iloc[0]['path']

'data/chest_xray/train/NORMAL/IM-0115-0001.jpeg'

In [24]:
from PIL import Image
import os 

class XrayDataset(Dataset):
    def __init__(self, split, transform):
        self.data = data[data['split'] == split]

        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        data_item = self.data.iloc[idx]
        label = self.data.iloc[idx]['class']
        image_data = Image.open(os.path.join("..", data_item['path']))
        image_data = self.transform(image_data)
        return image_data, label

In [25]:
train_transform = v2.Compose([
    v2.ToImage(),
    v2.Grayscale(num_output_channels=1),
    v2.ToDtype(torch.uint8, scale=True),
    v2.RandomResizedCrop(size=(500, 500)),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.Grayscale(num_output_channels=1),
    v2.ToDtype(torch.uint8, scale=True),
    v2.CenterCrop(size=(500, 500)),
    v2.ToDtype(torch.float32, scale=True),
])

In [26]:
train_dataset = XrayDataset(split='train', transform=train_transform)
val_dataset = XrayDataset(split="val", transform=test_transform)
test_dataset = XrayDataset(split="test", transform=test_transform)

train_dataloader = DataLoader(dataset=train_dataset, shuffle=True, batch_size=64)
val_dataloader = DataLoader(dataset=val_dataset, shuffle=False, batch_size=64)
test_dataloader = DataLoader(dataset=test_dataset, shuffle=False, batch_size=64)

In [27]:
device = torch.device('cuda') if torch.cuda.is_available else torch.device('cpu')

In [28]:
torch.cuda.is_available(), device

(True, device(type='cuda'))

In [29]:
class DenseLayer(nn.Module):
    def __init__(self, num_input_features, growth_rate):
        super().__init__()
        self.batchnorm1 = nn.BatchNorm2d(num_features=num_input_features)
        self.conv1x1 = nn.Conv2d(in_channels=num_input_features, out_channels=4*growth_rate, kernel_size=(1,1))
        self.batchnorm2 = nn.BatchNorm2d(num_features=4*growth_rate)
        self.conv3x3 = nn.Conv2d(in_channels=4*growth_rate, out_channels=growth_rate, kernel_size=(3,3), padding=1)

    def forward(self, previous_feature_maps):
        x = self.batchnorm1(previous_feature_maps)
        x = F.relu(x)
        x = self.conv1x1(x)
        x = self.batchnorm2(x)
        x = F.relu(x)
        x = self.conv3x3(x)
        return torch.cat((previous_feature_maps, x), dim=1)

In [30]:
class DenseBlock(nn.Module):
    def __init__(self, num_layers, num_input_features, growth_rate):
        super().__init__()
        input_features = num_input_features
        self.dense_layers = nn.ModuleList()
        for _ in range(num_layers):
            self.dense_layers.append(DenseLayer(num_input_features=input_features, growth_rate=growth_rate))
            input_features += growth_rate
    
    def forward(self, x):
        for layer in self.dense_layers:
            x = layer(x)
        
        return x

In [31]:
class TransitionLayer(nn.Module):
    def __init__(self, num_input_features, compression=0.5):
        super().__init__()
        self.batchnorm = nn.BatchNorm2d(num_features=num_input_features)
        self.conv1x1 = nn.Conv2d(in_channels=num_input_features, out_channels=math.floor(num_input_features * compression), kernel_size=(1,1))
        self.avgpool = nn.AvgPool2d(kernel_size=(2,2), stride=2)
    def forward(self, x):
        x = self.batchnorm(x)
        x = F.relu(x)
        x = self.conv1x1(x)
        x = self.avgpool(x)
        return x

In [32]:
class DenseNet121(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=(7,7), stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(num_features=64)
        self.maxpool = nn.MaxPool2d(kernel_size=(3,3), stride=2, padding=1)
        self.dense_blocks = nn.ModuleList()

        dense_block_layers = [6, 12, 24, 16]
        channel_count = 64
        for idx in range(4):
            block = DenseBlock(num_layers=dense_block_layers[idx], num_input_features=channel_count, growth_rate=32)
            channel_count = channel_count + 32 * dense_block_layers[idx]
            transition = TransitionLayer(num_input_features=channel_count, compression=0.5)
            self.dense_blocks.append(block)

            if idx != 3:
                channel_count =  channel_count // 2
                self.dense_blocks.append(transition)

        self.bn2 = nn.BatchNorm2d(channel_count)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(channel_count, num_classes)
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.maxpool(x)

        for block in self.dense_blocks:
            x = block(x)

        x = self.bn2(x)
        x = F.relu(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [33]:
model = DenseNet121(10)

In [34]:
summary(model=model, input_size=(16, 1, 500, 500))

Layer (type:depth-idx)                        Output Shape              Param #
DenseNet121                                   [16, 10]                  --
├─Conv2d: 1-1                                 [16, 64, 250, 250]        3,200
├─BatchNorm2d: 1-2                            [16, 64, 250, 250]        128
├─MaxPool2d: 1-3                              [16, 64, 125, 125]        --
├─ModuleList: 1-4                             --                        --
│    └─DenseBlock: 2-1                        [16, 256, 125, 125]       --
│    │    └─ModuleList: 3-1                   --                        336,000
│    └─TransitionLayer: 2-2                   [16, 128, 62, 62]         --
│    │    └─BatchNorm2d: 3-2                  [16, 256, 125, 125]       512
│    │    └─Conv2d: 3-3                       [16, 128, 125, 125]       32,896
│    │    └─AvgPool2d: 3-4                    [16, 128, 62, 62]         --
│    └─DenseBlock: 2-3                        [16, 512, 62, 62]         --
│    │

In [ ]:
epochs = 15
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), 0.01)

for epoch in range(epochs):
    epoch_loss = 0
    print(f"Epoch {epoch}:")
    model.train()
    for idx, (x, target) in enumerate(train_dataloader):
        input = x.to(device)
        target = target.to(device)
        output = model(input)
        output = output.squeeze()
        loss = criterion(output, target.float())
        # zero the gradients
        optimizer.zero_grad()
        # calculate the gradients based on the calculated loss (the gradient of the loss wrt each param)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    # Validation loss
    print("Loss", epoch_loss/len(train_dataloader))
    
    model.eval()
    with torch.no_grad():
        val_loss = 0
        all_preds = []
        all_targets = []

        for x, target in val_dataloader:
            x = x.to(device)
            target = target.to(device).float()

            output = model(x).squeeze()

            loss = criterion(output, target)
            val_loss += loss.item()

            preds = torch.sigmoid(output)
            all_preds.extend(preds.cpu())
            all_targets.extend(target.cpu())

        all_preds = torch.stack(all_preds)
        all_targets = torch.stack(all_targets)

        output_thresholded = (all_preds >= 0.5).numpy()
        accuracy = accuracy_score(all_targets.numpy(), output_thresholded)

        print(f"Val loss: {val_loss / len(val_dataloader)} | Val accuracy: {accuracy}")

Epoch 0:
